# P10.6-AI — Notebook 61: preflight subarticular Axial T2

Audita **estenosis subarticular izquierda y derecha** por nivel lumbar sobre **Axial T2** de RSNA/LumbarDISC y exporta el manifiesto candidato para el Notebook 62.

Este paso no entrena, no crea un internal test y no accede al test oficial. Se ejecuta con **CPU**.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


## Correcciones incorporadas

- Normaliza los niveles de coordenadas (`L1/L2`) al mismo formato de las etiquetas (`L1-L2`) **antes del merge**.
- Trata los labels faltantes de subarticular como exclusiones explícitas, no como una clase.
- Exige al menos 95 % de etiquetas conocidas y 97 % de coordenadas utilizables entre las etiquetas conocidas.
- Informa estratos raros o sin soporte sin bloquear el preflight cuando existe cobertura por lado, nivel y clases globales.


In [ ]:
# 1) Dependencias, Drive y rama
from __future__ import annotations

import importlib.util
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REQUIRED = {"numpy": "numpy", "pandas": "pandas"}
missing = [
    package
    for module, package in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing
    ])

import numpy as np
import pandas as pd
from google.colab import drive  # type: ignore

drive.mount("/content/drive", force_remount=False)

REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        ["git", "fetch", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "checkout", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "pull", "--ff-only", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

sys.path.insert(0, str(REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_preflight import (
    LEVELS,
    atomic_write_json,
    atomic_write_text,
    normalize_condition,
    normalize_level,
    normalize_series_description,
    normalize_severity,
    parse_label_column,
    sha256_file,
)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
DATA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
OUTPUT_ROOT = RESULTS_ROOT / "notebook61_subarticular_preflight"
NOTEBOOK60_SUMMARY = (
    RESULTS_ROOT
    / "notebook60_foraminal_evaluation"
    / "evaluation_summary.json"
)
INPUTS = {
    "train": DATA_ROOT / "train.csv",
    "coordinates": DATA_ROOT / "train_label_coordinates.csv",
    "series": DATA_ROOT / "train_series_descriptions.csv",
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

missing_inputs = [
    str(path)
    for path in [NOTEBOOK60_SUMMARY, *INPUTS.values()]
    if not path.is_file()
]
if missing_inputs:
    raise FileNotFoundError(
        "Faltan entradas:\n- " + "\n- ".join(missing_inputs)
    )

notebook60 = json.loads(
    NOTEBOOK60_SUMMARY.read_text(encoding="utf-8")
)
source60_gates = {
    "approved": notebook60.get("approved") is True,
    "status": (
        notebook60.get("status") == "APPROVED_FOR_NOTEBOOK_61"
    ),
    "nextNotebook": notebook60.get("nextNotebook") == 61,
    "finalModelExported": (
        notebook60.get("finalModel", {}).get("exported") is True
    ),
    "officialTestNotAccessed": (
        notebook60.get("governance", {})
        .get("officialTestAccessed") is False
    ),
}
if not all(source60_gates.values()):
    raise RuntimeError(
        "Notebook 60 no habilita Notebook 61: "
        + json.dumps(source60_gates, ensure_ascii=False)
    )

print({
    "repoRef": REPO_REF,
    "repoSha": REPO_SHA,
    "gpuRequired": False,
    "sourceNotebook60": source60_gates,
    "outputRoot": str(OUTPUT_ROOT),
})


In [ ]:
# 2) Cargar, validar y normalizar las etiquetas subarticulares
TARGET_CONDITIONS = (
    "subarticular_stenosis_left",
    "subarticular_stenosis_right",
)
SIDES = ("left", "right")
SEVERITY_CODE = {
    "normal_mild": 0,
    "moderate": 1,
    "severe": 2,
}
THRESHOLDS = {
    "minimumKnownLabelCoverage": 0.95,
    "minimumAxialT2StudyCoverage": 0.95,
    "minimumUsableCoordinateCoverage": 0.97,
    "expectedTargetColumns": 10,
}

train = pd.read_csv(
    INPUTS["train"],
    dtype={"study_id": "string"},
)
coordinates = pd.read_csv(
    INPUTS["coordinates"],
    dtype={
        "study_id": "string",
        "series_id": "string",
        "instance_number": "Int64",
    },
)
series = pd.read_csv(
    INPUTS["series"],
    dtype={
        "study_id": "string",
        "series_id": "string",
        "series_description": "string",
    },
)

schema_gates = {
    "train": {"study_id"} <= set(train.columns),
    "coordinates": {
        "study_id", "series_id", "instance_number",
        "condition", "level", "x", "y",
    } <= set(coordinates.columns),
    "series": {
        "study_id", "series_id", "series_description",
    } <= set(series.columns),
}
if not all(schema_gates.values()):
    raise RuntimeError(
        "Esquema inesperado: "
        + json.dumps(schema_gates, ensure_ascii=False)
    )

for frame, columns in [
    (train, ["study_id"]),
    (coordinates, ["study_id", "series_id"]),
    (series, ["study_id", "series_id"]),
]:
    for column in columns:
        frame[column] = (
            frame[column].astype("string").str.strip()
        )

if train["study_id"].isna().any():
    raise RuntimeError("train.csv contiene study_id nulos.")
if train["study_id"].duplicated().any():
    raise RuntimeError("train.csv contiene study_id duplicados.")

series_conflicts = int(
    (
        series.groupby(["study_id", "series_id"])
        ["series_description"]
        .nunique(dropna=False)
        > 1
    ).sum()
)
series = (
    series
    .sort_values(["study_id", "series_id", "series_description"])
    .drop_duplicates(["study_id", "series_id"], keep="first")
    .reset_index(drop=True)
)

target_columns = []
for column in train.columns:
    if column == "study_id":
        continue
    condition, level = parse_label_column(str(column))
    if condition in TARGET_CONDITIONS and level in LEVELS:
        side = condition.rsplit("_", 1)[-1]
        target_columns.append(
            (str(column), condition, side, level)
        )

expected_pairs = {
    (side, level)
    for side in SIDES
    for level in LEVELS
}
observed_pairs = {
    (side, level)
    for _, _, side, level in target_columns
}
if observed_pairs != expected_pairs:
    raise RuntimeError({
        "missingTargets": sorted(expected_pairs - observed_pairs),
        "unexpectedTargets": sorted(observed_pairs - expected_pairs),
    })

label_frames = []
for column, condition, side, level in target_columns:
    frame = (
        train[["study_id", column]]
        .rename(columns={column: "severity_raw"})
        .copy()
    )
    frame["label_column"] = column
    frame["condition"] = condition
    frame["side"] = side
    frame["level"] = level
    frame["severity"] = frame["severity_raw"].map(
        normalize_severity
    )
    frame["severity_code"] = (
        frame["severity"].map(SEVERITY_CODE).astype("Int64")
    )
    label_frames.append(frame)

labels = (
    pd.concat(label_frames, ignore_index=True)
    .sort_values(["study_id", "side", "level"])
    .reset_index(drop=True)
)
if labels[["study_id", "side", "level"]].duplicated().any():
    raise RuntimeError(
        "Hay claves study_id-side-level duplicadas en etiquetas."
    )

known_label_count = int(labels["severity"].notna().sum())
missing_label_count = int(labels["severity"].isna().sum())
known_label_coverage = (
    known_label_count / len(labels) if len(labels) else 0.0
)

label_distribution = (
    labels
    .groupby(
        ["side", "level", "severity"],
        dropna=False,
    )
    .size()
    .reset_index(name="rows")
)
print({
    "targetColumns": len(target_columns),
    "labelRows": int(len(labels)),
    "knownLabels": known_label_count,
    "missingLabelsExplicitlyExcluded": missing_label_count,
    "knownLabelCoverage": round(known_label_coverage, 6),
})
display(label_distribution)


In [ ]:
# 3) Inventario Axial T2, coordenadas normalizadas y manifiesto candidato
description_parts = series["series_description"].map(
    normalize_series_description
)
series_normalized = series.copy()
series_normalized["plane"] = description_parts.map(
    lambda item: item["plane"]
)
series_normalized["sequence"] = description_parts.map(
    lambda item: item["sequence"]
)
series_normalized["sequence_category"] = description_parts.map(
    lambda item: item["sequenceCategory"]
)

axial_t2 = series_normalized.loc[
    series_normalized["sequence_category"].eq("axial_t2")
].copy()

known_study_ids = set(
    labels.loc[labels["severity"].notna(), "study_id"]
)
axial_studies = int(
    axial_t2.loc[
        axial_t2["study_id"].isin(known_study_ids),
        "study_id",
    ].nunique()
)
known_studies = int(len(known_study_ids))
axial_t2_study_coverage = (
    axial_studies / known_studies if known_studies else 0.0
)

coordinate_frame = coordinates.copy()
coordinate_frame["condition_normalized"] = (
    coordinate_frame["condition"].map(normalize_condition)
)
coordinate_frame["level_raw"] = coordinate_frame["level"]
coordinate_frame["level"] = (
    coordinate_frame["level"].map(normalize_level)
)
coordinate_frame = coordinate_frame.loc[
    coordinate_frame["condition_normalized"]
    .isin(TARGET_CONDITIONS)
    & coordinate_frame["level"].isin(LEVELS)
].copy()
coordinate_frame["side"] = (
    coordinate_frame["condition_normalized"]
    .str.rsplit("_", n=1)
    .str[-1]
)

for column in ["instance_number", "x", "y"]:
    coordinate_frame[column] = pd.to_numeric(
        coordinate_frame[column],
        errors="coerce",
    )

coordinate_frame = coordinate_frame.merge(
    series_normalized[[
        "study_id",
        "series_id",
        "series_description",
        "sequence_category",
    ]],
    on=["study_id", "series_id"],
    how="left",
    validate="many_to_one",
)

coordinate_frame["coordinate_numeric"] = (
    coordinate_frame[["instance_number", "x", "y"]]
    .notna()
    .all(axis=1)
)
coordinate_frame["coordinate_nonnegative"] = (
    coordinate_frame["instance_number"].fillna(-1).ge(0)
    & coordinate_frame["x"].fillna(-1).ge(0)
    & coordinate_frame["y"].fillna(-1).ge(0)
)
coordinate_frame["coordinate_on_axial_t2"] = (
    coordinate_frame["sequence_category"].eq("axial_t2")
)
coordinate_frame["valid_subarticular_coordinate"] = (
    coordinate_frame["coordinate_on_axial_t2"]
    & coordinate_frame["coordinate_numeric"]
    & coordinate_frame["coordinate_nonnegative"]
    & coordinate_frame["level"].isin(LEVELS)
)

valid_coordinates = (
    coordinate_frame.loc[
        coordinate_frame["valid_subarticular_coordinate"]
    ]
    .sort_values([
        "study_id",
        "side",
        "level",
        "series_id",
        "instance_number",
    ])
    .copy()
)
candidate_counts = (
    valid_coordinates
    .groupby(["study_id", "side", "level"])
    .size()
    .reset_index(name="valid_coordinate_candidates")
)
selected_coordinates = (
    valid_coordinates
    .drop_duplicates(
        ["study_id", "side", "level"],
        keep="first",
    )
    [[
        "study_id",
        "side",
        "level",
        "condition_normalized",
        "series_id",
        "instance_number",
        "x",
        "y",
        "series_description",
        "sequence_category",
    ]]
    .rename(columns={
        "condition_normalized": "coordinate_condition",
        "series_id": "coordinate_series_id",
        "instance_number": "coordinate_instance_number",
        "x": "coordinate_x",
        "y": "coordinate_y",
        "series_description": "coordinate_series_description",
    })
    .merge(
        candidate_counts,
        on=["study_id", "side", "level"],
        how="left",
        validate="one_to_one",
    )
)
duplicate_coordinate_candidates = valid_coordinates.merge(
    candidate_counts.loc[
        candidate_counts["valid_coordinate_candidates"] > 1
    ],
    on=["study_id", "side", "level"],
    how="inner",
)

coordinate_status = (
    coordinate_frame
    .groupby(["study_id", "side", "level"])
    .agg(
        coordinate_rows=("series_id", "size"),
        axial_t2_rows=(
            "coordinate_on_axial_t2",
            "sum",
        ),
        numeric_rows=("coordinate_numeric", "sum"),
        valid_rows=(
            "valid_subarticular_coordinate",
            "sum",
        ),
    )
    .reset_index()
)

audit = (
    labels
    .merge(
        selected_coordinates,
        on=["study_id", "side", "level"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        coordinate_status,
        on=["study_id", "side", "level"],
        how="left",
        validate="one_to_one",
    )
)
for column in [
    "coordinate_rows",
    "axial_t2_rows",
    "numeric_rows",
    "valid_rows",
]:
    audit[column] = audit[column].fillna(0).astype(int)

def exclusion_reason(row) -> str:
    if pd.isna(row["severity"]):
        return "missing_label"
    if int(row["valid_rows"]) > 0:
        return ""
    if int(row["coordinate_rows"]) == 0:
        return "missing_coordinate"
    if int(row["axial_t2_rows"]) == 0:
        return "coordinate_not_axial_t2"
    if int(row["numeric_rows"]) == 0:
        return "invalid_coordinate_values"
    return "no_selectable_coordinate"

audit["exclusion_reason"] = audit.apply(
    exclusion_reason,
    axis=1,
)
audit["eligible"] = audit["exclusion_reason"].eq("")

manifest_columns = [
    "study_id",
    "condition",
    "side",
    "level",
    "severity",
    "severity_code",
    "label_column",
    "severity_raw",
    "coordinate_condition",
    "coordinate_series_id",
    "coordinate_instance_number",
    "coordinate_x",
    "coordinate_y",
    "coordinate_series_description",
    "sequence_category",
    "valid_coordinate_candidates",
]
manifest = (
    audit.loc[audit["eligible"], manifest_columns]
    .sort_values(["study_id", "side", "level"])
    .reset_index(drop=True)
)
manifest["severity_code"] = (
    manifest["severity_code"].astype(int)
)
manifest["coordinate_instance_number"] = (
    manifest["coordinate_instance_number"].round().astype(int)
)
manifest["human_review_required"] = True
manifest["not_clinical_diagnosis"] = True
manifest["official_test_accessed"] = False
manifest["source_notebook"] = 61

exclusions = (
    audit.loc[~audit["eligible"]]
    .sort_values(["study_id", "side", "level"])
    .reset_index(drop=True)
)
manifest_distribution = (
    manifest
    .groupby(["side", "level", "severity"])
    .size()
    .reset_index(name="rows")
)
exclusion_distribution = (
    exclusions
    .groupby(["exclusion_reason"])
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

usable_coordinate_coverage = (
    len(manifest) / known_label_count
    if known_label_count
    else 0.0
)
print({
    "subarticularCoordinateRows": int(len(coordinate_frame)),
    "validAxialT2CoordinateRows": int(len(valid_coordinates)),
    "manifestRows": int(len(manifest)),
    "manifestStudies": int(manifest["study_id"].nunique()),
    "usableCoordinateCoverageAmongKnownLabels": round(
        usable_coordinate_coverage,
        6,
    ),
    "duplicateCoordinateCandidateRows": int(
        len(duplicate_coordinate_candidates)
    ),
})
display(exclusion_distribution)
display(manifest_distribution)


In [ ]:
# 4) Gates, evidencia y exportación
expected_index = pd.MultiIndex.from_product(
    [SIDES, LEVELS, tuple(SEVERITY_CODE)],
    names=["side", "level", "severity"],
)
stratum_support = (
    manifest_distribution
    .set_index(["side", "level", "severity"])["rows"]
    .reindex(expected_index, fill_value=0)
    .rename("rows")
    .reset_index()
)
side_level_support = (
    manifest
    .groupby(["side", "level"])
    .size()
    .reindex(
        pd.MultiIndex.from_product(
            [SIDES, LEVELS],
            names=["side", "level"],
        ),
        fill_value=0,
    )
)
classes_by_side = {
    side: set(
        manifest.loc[
            manifest["side"].eq(side),
            "severity",
        ]
    )
    for side in SIDES
}

finite_coordinates = bool(
    not manifest.empty
    and np.isfinite(
        manifest[[
            "coordinate_instance_number",
            "coordinate_x",
            "coordinate_y",
        ]].to_numpy(dtype=float)
    ).all()
)
process_gates = {
    "sourceNotebook60Approved": all(
        source60_gates.values()
    ),
    "requiredSchema": all(schema_gates.values()),
    "targetColumnsComplete": (
        len(target_columns)
        == THRESHOLDS["expectedTargetColumns"]
    ),
    "seriesMetadataConsistent": (
        series_conflicts == 0
    ),
    "knownLabelCoverage": (
        known_label_coverage
        >= THRESHOLDS["minimumKnownLabelCoverage"]
    ),
    "missingLabelsExplicitlyExcluded": (
        not manifest["severity"].isna().any()
    ),
    "axialT2StudyCoverage": (
        axial_t2_study_coverage
        >= THRESHOLDS["minimumAxialT2StudyCoverage"]
    ),
    "usableCoordinateCoverage": (
        usable_coordinate_coverage
        >= THRESHOLDS["minimumUsableCoordinateCoverage"]
    ),
    "uniqueStudySideLevelKeys": (
        not manifest[[
            "study_id", "side", "level",
        ]].duplicated().any()
    ),
    "axialT2Only": (
        not manifest.empty
        and manifest["sequence_category"].eq(
            "axial_t2"
        ).all()
    ),
    "finiteCoordinates": finite_coordinates,
    "allGlobalClassesPresent": (
        set(manifest["severity"]) == set(SEVERITY_CODE)
    ),
    "allClassesPresentBySide": all(
        values == set(SEVERITY_CODE)
        for values in classes_by_side.values()
    ),
    "allSideLevelPairsPresent": bool(
        (side_level_support > 0).all()
    ),
    "manifestNotEmpty": not manifest.empty,
    "officialTestNotAccessed": True,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
}
approved = all(process_gates.values())
status = (
    "APPROVED_FOR_NOTEBOOK_62"
    if approved
    else "SUBARTICULAR_PREFLIGHT_REVIEW_REQUIRED"
)

study_summary = (
    audit.groupby("study_id")
    .agg(
        expected_targets=("study_id", "size"),
        known_labels=(
            "severity",
            lambda values: int(values.notna().sum()),
        ),
        eligible_targets=("eligible", "sum"),
    )
    .reset_index()
)
study_summary["complete_ten_targets"] = (
    study_summary["eligible_targets"].eq(10)
)

paths = {
    "manifest": (
        OUTPUT_ROOT
        / "subarticular_candidate_manifest.csv"
    ),
    "exclusions": (
        OUTPUT_ROOT
        / "subarticular_exclusions.csv"
    ),
    "labelDistribution": (
        OUTPUT_ROOT
        / "subarticular_label_distribution.csv"
    ),
    "manifestDistribution": (
        OUTPUT_ROOT
        / "subarticular_manifest_distribution.csv"
    ),
    "stratumSupport": (
        OUTPUT_ROOT
        / "subarticular_stratum_support.csv"
    ),
    "coordinateAudit": (
        OUTPUT_ROOT
        / "subarticular_coordinate_audit.csv"
    ),
    "duplicateCandidates": (
        OUTPUT_ROOT
        / "subarticular_duplicate_coordinate_candidates.csv"
    ),
    "studySummary": (
        OUTPUT_ROOT
        / "subarticular_study_summary.csv"
    ),
    "axialInventory": (
        OUTPUT_ROOT
        / "axial_t2_series_inventory.csv"
    ),
    "report": (
        OUTPUT_ROOT
        / "subarticular_preflight_report.md"
    ),
    "summary": (
        OUTPUT_ROOT
        / "subarticular_preflight_summary.json"
    ),
}
for key, frame in {
    "manifest": manifest,
    "exclusions": exclusions,
    "labelDistribution": label_distribution,
    "manifestDistribution": manifest_distribution,
    "stratumSupport": stratum_support,
    "coordinateAudit": coordinate_frame,
    "duplicateCandidates": duplicate_coordinate_candidates,
    "studySummary": study_summary,
    "axialInventory": axial_t2,
}.items():
    frame.to_csv(paths[key], index=False)

report_lines = [
    "# P10.6-AI — Preflight subarticular Axial T2",
    "",
    f"- Estado: `{status}`",
    f"- Etiquetas conocidas: {known_label_count}/{len(labels)} "
    f"({known_label_coverage:.4f})",
    f"- Cobertura de estudios con Axial T2: "
    f"{axial_t2_study_coverage:.4f}",
    f"- Cobertura de coordenadas entre labels conocidos: "
    f"{usable_coordinate_coverage:.4f}",
    f"- Filas candidatas: {len(manifest)}",
    f"- Estudios candidatos: "
    f"{manifest['study_id'].nunique()}",
    "",
    "## Gates",
    "",
]
report_lines.extend(
    f"- {name}: `{str(bool(value)).lower()}`"
    for name, value in process_gates.items()
)
report_lines.extend([
    "",
    "Los labels faltantes se excluyen y no se imputan.",
    "Los estratos con soporte cero quedan informados en "
    "`subarticular_stratum_support.csv`; el Notebook 62 "
    "definirá reglas de split según soporte.",
    "No se accedió al test oficial ni se creó un internal test.",
    "",
])
atomic_write_text(
    paths["report"],
    "\n".join(report_lines),
)

summary = {
    "schemaVersion": (
        "pfi.rsna-subarticular-preflight.v2"
    ),
    "ticket": "P10.6-AI",
    "notebook": 61,
    "sourceNotebook": 60,
    "createdAtUtc": datetime.now(
        timezone.utc
    ).isoformat(),
    "repoRef": REPO_REF,
    "repoSha": REPO_SHA,
    "dataset": "RSNA_LumbarDISC",
    "task": "subarticular_stenosis_left_right",
    "sequence": "Axial T2",
    "status": status,
    "approved": approved,
    "nextNotebook": 62 if approved else None,
    "sourceNotebook60": {
        "sha256": sha256_file(NOTEBOOK60_SUMMARY),
        "gates": source60_gates,
    },
    "thresholds": THRESHOLDS,
    "data": {
        "studies": int(train["study_id"].nunique()),
        "labelRows": int(len(labels)),
        "knownLabelRows": known_label_count,
        "missingLabelRows": missing_label_count,
        "candidateRows": int(len(manifest)),
        "candidateStudies": int(
            manifest["study_id"].nunique()
        ),
        "excludedRows": int(len(exclusions)),
        "axialT2Series": int(len(axial_t2)),
        "completeTenTargetStudies": int(
            study_summary["complete_ten_targets"].sum()
        ),
        "zeroSupportStrata": int(
            (stratum_support["rows"] == 0).sum()
        ),
    },
    "coverage": {
        "knownLabelCoverage": known_label_coverage,
        "axialT2StudyCoverage": (
            axial_t2_study_coverage
        ),
        "usableCoordinateCoverage": (
            usable_coordinate_coverage
        ),
    },
    "inputSha256": {
        key: sha256_file(path)
        for key, path in INPUTS.items()
    },
    "gateResults": process_gates,
    "governance": {
        "humanReviewRequired": True,
        "notClinicalDiagnosis": True,
        "officialTestAccessed": False,
        "internalTestAccessed": False,
        "internalTestSealed": False,
        "missingLabelsImputed": False,
    },
}
summary["outputSha256"] = {
    key: sha256_file(path)
    for key, path in paths.items()
    if key != "summary" and path.is_file()
}
atomic_write_json(paths["summary"], summary)

print({
    "status": status,
    "approved": approved,
    "nextNotebook": 62 if approved else None,
    "gateResults": process_gates,
    "outputs": sorted(
        path.name for path in paths.values()
    ),
})
if not approved:
    raise RuntimeError(
        "Revisar subarticular_preflight_summary.json "
        "antes de continuar."
    )


## Resultado esperado

Una ejecución aprobada termina con:

```text
APPROVED_FOR_NOTEBOOK_62
```

El Notebook 62 realizará el split por `study_id`, documentará los estratos raros y sellará el internal test.
